# Fantasy Trade Calculator

A dynasty/redraft fantasy football trade calculator that trains supervised gradient-boosting
models to predict market consensus value (FantasyCalc, blended with KeepTradeCut) from player
fundamentals, producing one unified value board per format that puts every player and draft pick
on the same directly-comparable scale.

**Consensus value is the training target, not a benchmark to beat.** These models are trained to
reflect the fantasy trade market as closely as possible -- agreement with FantasyCalc/KeepTradeCut
consensus is the design goal, not an independent finding. Where a player's model-predicted value
diverges from their current market price, that divergence is reported as a residual: a candidate
worth investigating, never a claim that the model is right and the market is wrong.

Full methodology, scope, and known exclusions: see `README.md` in this directory.

This notebook is a thin orchestration layer -- all real logic lives in
`trade_calculator_pipeline.py`, which is imported and called stage by stage below.

In [1]:
import pandas as pd

import trade_calculator_pipeline as tcp

pd.set_option("display.max_rows", 250)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

## Stage 1 -- ID crosswalk

Built from nflreadpy's `load_ff_playerids()`, the backbone every source join uses.

In [2]:
crosswalk = tcp.build_crosswalk()
print(f"crosswalk rows: {len(crosswalk)}")
crosswalk.head()

crosswalk rows: 12472


,mfl_id,sleeper_id,espn_id,gsis_id,ktc_id,pfr_id,name,position,merge_name
0,17462,13269.0,4837248.0,00-0041562,1924.0,MendFe00,Fernando Mendoza,QB,fernando mendoza
1,17463,13275.0,4685522.0,00-0041568,1925.0,SimpTy00,Ty Simpson,QB,ty simpson
2,17464,NaN,NaN,NaN,NaN,NaN,Trinidad Chambliss,XX,trinidad chambliss
3,17465,13404.0,4567747.0,00-0040906,1927.0,NussGa00,Garrett Nussmeier,QB,garrett nussmeier
4,17466,13272.0,4430841.0,00-0041561,1928.0,BeckCa01,Carson Beck,QB,carson beck


## Stage 2 -- Pull FantasyCalc (6 format-teamsize combos) and KeepTradeCut (dynasty + redraft)

In [3]:
fc_all = tcp.pull_all_fantasycalc()
for k, v in fc_all.items():
    print(f"  {k}: {len(v)} rows")

ktc_dyn = tcp.pull_ktc_dynasty()
ktc_redraft = tcp.pull_ktc_redraft()
print(f"ktc_dynasty: {len(ktc_dyn)} rows, ktc_redraft: {len(ktc_redraft)} rows")

  ('sf_dynasty', 10): 475 rows
  ('sf_dynasty', 12): 475 rows
  ('oneqb_dynasty', 10): 475 rows
  ('oneqb_dynasty', 12): 475 rows
  ('redraft', 10): 171 rows
  ('redraft', 12): 171 rows


ktc_dynasty: 500 rows, ktc_redraft: 300 rows


### Blend example -- SF dynasty, 12 team

A quick look at one format's crosswalk join + percentile blend before running the full 6-combo
version in Stage 7.

In [4]:
fc_joined_demo = tcp.join_fantasycalc_crosswalk(fc_all[("sf_dynasty", 12)], crosswalk, label="FC-SF12")
ktc_joined_demo = tcp.join_ktc_crosswalk(ktc_dyn, crosswalk, label="KTC-dyn")
blended_demo = tcp.blend_consensus(fc_joined_demo, ktc_joined_demo, ktc_value_col="sf_value")
blended_demo[["name", "position", "value", "ktc_percentile", "fc_percentile", "consensus_value"]] \
    .sort_values("consensus_value", ascending=False).head(15)

[FC-SF12] 1 rows matched via normalized-name fallback (documented last resort)
[FC-SF12] 1 unmatched after fallback: ['Matt Hibner']
[FC-SF12] top-200 join coverage: 100.0%
[KTC-dyn] 7 unmatched after fallback: ['Hassan Haskins', 'Stetson Bennett', 'Dylan Laube', 'Thomas Fidone', 'Sterling Shepard', 'Foster Moreau', 'Tim Patrick']
[KTC-dyn] top-200 join coverage: 100.0%


,name,position,value,ktc_percentile,fc_percentile,consensus_value
0,Bijan Robinson,RB,10438,99.890591,100.000000,10405.618873
1,Josh Allen,QB,10225,99.562363,99.748744,10215.767328
2,Jahmyr Gibbs,RB,10189,99.890591,99.497487,10208.844721
3,Ja'Marr Chase,WR,10080,99.343545,99.246231,10095.560303
4,Jaxon Smith-Njigba,WR,8777,99.124726,98.994975,9025.013154
5,Drake Maye,QB,8595,98.905908,98.743719,8638.302373
6,Puka Nacua,WR,8308,98.687090,98.492462,8389.941414
7,Brock Bowers,TE,7606,98.468271,98.241206,7839.832817
9,Amon-Ra St. Brown,WR,7264,98.249453,97.738693,7288.166341
8,Lamar Jackson,QB,7297,97.592998,97.989950,7279.461681


## Stage 3 -- Feature assembly

Draft capital, season-level production/efficiency, continuous position-specific aging curves, and
combine metrics (RB speed score).

In [5]:
draft_capital = tcp.load_draft_capital(crosswalk)
print(f"draft_capital rows: {len(draft_capital)}")

weekly = tcp.load_weekly_features()
print(f"weekly/season feature rows: {len(weekly)}, cols: {len(weekly.columns)}")

curve_params, peak_ages, aged = tcp.fit_aging_curves(weekly, draft_capital)
print(f"peak ages by position: { {k: round(v, 1) for k, v in peak_ages.items()} }")

combine_feats = tcp.load_combine_features()
print(f"combine feature rows: {len(combine_feats)}")
combine_feats.dropna(subset=["speed_score"]).head(3)

[draft_capital] 2 duplicate gsis_id rows in load_draft_picks -- keeping first (data-entry dupes, e.g. supplemental draft)
[draft_capital] 73 rows had a non-gsis id from load_draft_picks (new/unplayed draft class) -- 73/73 repaired via crosswalk name match
[draft_capital] 1 rows still unresolved after repair attempt: ['Mike Washington Jr.']
draft_capital rows: 3551


[snap_share] 1.0% of player-season snap rows unmatched to gsis_id via pfr_id (dropped)
weekly/season feature rows: 4191, cols: 27
[aging_curve] 1261 / 4191 player-seasons have no draft-capital age match (undrafted players) -- age kept as NaN, excluded only from curve fitting
peak ages by position: {'QB': 27.0, 'RB': np.float64(26.0), 'WR': np.float64(27.2), 'TE': np.float64(32.2)}


[combine] 35.1% of combine rows unmatched to gsis_id via pfr_id
combine feature rows: 1954


,gsis_id,forty_time,combine_weight,combine_height,speed_score
56,00-0019641,4.45,216.0,5-10,110.165016
118,00-0020514,4.38,207.0,5-9,112.487405
125,00-0020270,4.53,226.0,6-0,107.336054


### Rookie college-production features (CollegeFootballData)

Dominator rating (80% yardage / 20% TD weighted share of team receiving output) and breakout age
for the current draft class. **Note:** CFBD's free-tier monthly call quota can be exhausted at
build time -- if so, this degrades gracefully to `NaN` for affected rookies, who then fall back to
draft capital + age + combine speed score alone (`HistGradientBoostingRegressor` handles missing
features natively; see README's "Known exclusions").

In [6]:
rookie_college = tcp.compute_rookie_college_features(draft_capital, draft_year=2026)
coverage = (rookie_college["college_seasons_found"] > 0).mean()
print(f"2026 rookie CFBD college-production coverage: {coverage:.1%}"
      + ("  (degraded -- CFBD quota likely exhausted this cycle)" if coverage < 0.3 else ""))
rookie_college.sort_values("dominator_rating", ascending=False).head(10)

[cfbd_rookie] 0.0% of 2026 skill draftees matched to >=1 CFBD college season with usable receiving-dominator data (QBs excluded from this stat by design)
2026 rookie CFBD college-production coverage: 0.0%  (degraded -- CFBD quota likely exhausted this cycle)


,gsis_id,dominator_rating,breakout_age,college_seasons_found
0,00-0041562,NaN,NaN,0
1,00-0041027,NaN,NaN,0
2,00-0041438,NaN,NaN,0
3,00-0041029,NaN,NaN,0
4,00-0041568,NaN,NaN,0
5,00-0041032,NaN,NaN,0
6,00-0040867,NaN,NaN,0
7,00-0041547,NaN,NaN,0
8,00-0041511,NaN,NaN,0
9,00-0041512,NaN,NaN,0


## Stage 4 -- Draft pick valuation

2027-2029 x rounds 1-3 x {Early, Mid, Late} tiers, priced from native FantasyCalc/KeepTradeCut
market data where available, constructed via a year-discount/class-strength/format-multiplier
formula elsewhere. Redraft formats have no pick assets.

In [7]:
pick_universe = tcp.build_pick_universe(fc_all, ktc_dyn)
print(f"pick_universe rows: {len(pick_universe)}")
pick_universe[(pick_universe["format"] == "sf_dynasty") & (pick_universe["team_size"] == 12)] \
    .sort_values("pick_value", ascending=False)[["year", "round", "tier", "fc_value", "pick_value"]]

[format_multiplier] overall mean SF/1QB pick-value ratio: 1.060
[format_multiplier] by round: {1: 1.061, 2: 1.061, 3: 1.06, 4: 1.059}
[format_multiplier] NOTE: spec §6 cites a 30-60% player-level SF premium for elite QBs; picks came out much lower (~6%). That's expected, not a bug -- a pick is position-agnostic (might become a WR/RB/etc.), so it can't carry a specific elite-QB premium the way a known franchise QB player does.


[pick_universe] 36 native FC tier prices, 72 FC-derived (aggregate x tier-ratio), 0 fully constructed (of 108 total)
pick_universe rows: 108


,year,round,tier,fc_value,pick_value
27,2027,1,Early,4532.000000,5389.500000
36,2028,1,Early,3190.750270,3885.937676
28,2027,1,Mid,2984.000000,3862.850000
29,2027,1,Late,2297.000000,3237.800000
37,2028,1,Mid,2101.090720,2966.258968
45,2029,1,Early,2910.832307,2910.832307
30,2027,2,Early,1843.000000,2496.450000
38,2028,1,Late,1617.233964,2476.402076
31,2027,2,Mid,1528.000000,2179.350000
39,2028,2,Early,1583.333802,2142.166971


## Stage 5 -- Unified feature table

One row per player, "entering the 2026 season" snapshot: most-recent-season production plus
career aggregates (kept separate from single-season stats and from age -- never blended into one
composite score), current age, draft capital, combine metrics, and rookie college-production
features all in one table keyed on `gsis_id`.

In [8]:
snapshot = tcp.build_player_snapshot(aged, draft_capital, peak_ages)
feature_table = tcp.build_feature_table(snapshot, draft_capital, combine_feats, rookie_college)
feature_table[["gsis_id", "position", "current_age", "career_games", "ppg_ppr"]].head()

[snapshot] 1422 player rows (73 zero-snap 2026 rookies appended with NaN production/career features)


[feature_table] 1422 rows, 43 columns


,gsis_id,position,current_age,career_games,ppg_ppr
0,00-0019596,QB,49.0,66.0,15.980000
1,00-0020531,QB,47.0,23.0,17.456667
2,00-0021206,QB,47.0,2.0,0.380000
3,00-0022127,TE,44.0,26.0,3.190000
4,00-0022787,QB,45.0,6.0,-0.400000


## Stage 6 -- Consensus targets for all 6 format-teamsize combos

In [9]:
consensus_targets = tcp.build_all_consensus_targets(fc_all, ktc_dyn, ktc_redraft, crosswalk)
for k, v in consensus_targets.items():
    print(f"  {k}: {len(v)} players with a consensus target")

[KTC-dyn] 7 unmatched after fallback: ['Hassan Haskins', 'Stetson Bennett', 'Dylan Laube', 'Thomas Fidone', 'Sterling Shepard', 'Foster Moreau', 'Tim Patrick']
[KTC-dyn] top-200 join coverage: 100.0%
[ktc_redraft_bridge] 281/300 rows bridged to dynasty-page playerID by name (93.7%)


[KTC-redraft] 14 unmatched after fallback: ['Joe Mixon', 'Nick Chubb', 'Miles Sanders', 'DeAndre Hopkins', 'Adam Thielen', 'Jermaine Burton', 'Tyler Lockett', 'Hunter Renfrow', 'Austin Ekeler', 'Brandin Cooks', 'Zay Jones', 'Taysom Hill', 'Russell Wilson', 'Raheem Mostert']
[KTC-redraft] top-200 join coverage: 99.5%


[FC-sf_dynasty10] 1 rows matched via normalized-name fallback (documented last resort)
[FC-sf_dynasty10] 1 unmatched after fallback: ['Matt Hibner']
[FC-sf_dynasty10] top-200 join coverage: 100.0%


[FC-sf_dynasty12] 1 rows matched via normalized-name fallback (documented last resort)
[FC-sf_dynasty12] 1 unmatched after fallback: ['Matt Hibner']
[FC-sf_dynasty12] top-200 join coverage: 100.0%


[FC-oneqb_dynasty10] 1 rows matched via normalized-name fallback (documented last resort)


[FC-oneqb_dynasty10] 1 unmatched after fallback: ['Matt Hibner']
[FC-oneqb_dynasty10] top-200 join coverage: 100.0%


[FC-oneqb_dynasty12] 1 rows matched via normalized-name fallback (documented last resort)
[FC-oneqb_dynasty12] 1 unmatched after fallback: ['Matt Hibner']
[FC-oneqb_dynasty12] top-200 join coverage: 100.0%


[FC-redraft10] top-200 join coverage: 100.0%


[FC-redraft12] top-200 join coverage: 100.0%
  ('sf_dynasty', 10): 398 players with a consensus target
  ('sf_dynasty', 12): 398 players with a consensus target
  ('oneqb_dynasty', 10): 398 players with a consensus target
  ('oneqb_dynasty', 12): 398 players with a consensus target
  ('redraft', 10): 171 players with a consensus target
  ('redraft', 12): 171 players with a consensus target


## Stage 7 -- Fit all 6 models

One `HistGradientBoostingRegressor` per format-teamsize combo, trained on `log(consensus_value)`
with `position` as a native categorical feature, validated via 10-fold out-of-fold cross-validated
predictions (not train-set fit metrics). A `RandomForestRegressor` (imputed/encoded) serves as a
comparison baseline. Zero-snap rookies get an auxiliary lower-quantile prediction as their final
value instead of the mean -- a genuine predictive-uncertainty discount, never a neutral risk
multiplier.

**Sample-weighted training.** `HistGradientBoostingRegressor`'s default `min_samples_leaf=20`,
combined with this project's ~379-row training pool, structurally forces the top ~15-20 "elite tier"
players -- in any position, any format -- into leaves blended with lower-value neighbors, since every
leaf must average at least 20 samples. That systematically compresses predictions for the most
valuable assets (first surfaced as superflex elite-QB underprediction -- Josh Allen, Lamar Jackson,
Jayden Daniels were predicted well below their own training target -- then confirmed as a general,
position-agnostic effect across the whole board). The fix: every training row is weighted by its own
percentile rank in that format's `consensus_value` distribution (`weight = 1 + percentile**3`,
purely a function of where a row sits in its own format's target -- no position or format is
special-cased anywhere), combined with a lower `min_samples_leaf` (10). Verified across all 6 formats
with a multi-seed robustness check: this fixes the elite-QB compression as well as an earlier
QB-only board-assembly blend did, *and* also fixes the smaller RB/WR/TE top-of-board compression that
blend never touched, with Spearman/MAE flat-to-improved everywhere -- not a tradeoff against overall
accuracy. See `trade_calculator_pipeline.py`'s module comment above `fit_format_model` for the full
diagnosis, including the two earlier fixes (a QB-only model split, a starter-status feature) that
were tried and honestly ruled out first.

In [10]:
fit_results = tcp.fit_all_format_models(feature_table, consensus_targets)

validation_rows = []
for (fmt, team_size), res in fit_results.items():
    hm, rm = res["hgb_metrics"], res["rf_metrics"]
    validation_rows.append({
        "format": fmt, "team_size": team_size,
        "hgb_spearman": hm["spearman"], "hgb_mae": hm["mae"], "hgb_top50_mae": hm["top50_mae"],
        "rf_spearman": rm["spearman"], "rf_mae": rm["mae"],
    })
validation_df = pd.DataFrame(validation_rows)
validation_df

[sf_dynasty-10] dropping 2 feature(s) with <2 distinct values in this group: ['breakout_age', 'dominator_rating']


[sf_dynasty-10] rookie quantile discount: 55 rookies in training pool, mean-model vs quantile-model prediction reduced by 7.4% on average
[sf_dynasty-10] n=379  HGBR: spearman=0.826 mae=649 top50_mae=1574  |  RF baseline: spearman=0.827 mae=655 top50_mae=1816
[sf_dynasty-12] dropping 2 feature(s) with <2 distinct values in this group: ['breakout_age', 'dominator_rating']


[sf_dynasty-12] rookie quantile discount: 55 rookies in training pool, mean-model vs quantile-model prediction reduced by 8.4% on average
[sf_dynasty-12] n=379  HGBR: spearman=0.826 mae=671 top50_mae=1694  |  RF baseline: spearman=0.830 mae=659 top50_mae=1834
[oneqb_dynasty-10] dropping 2 feature(s) with <2 distinct values in this group: ['breakout_age', 'dominator_rating']


[oneqb_dynasty-10] rookie quantile discount: 55 rookies in training pool, mean-model vs quantile-model prediction reduced by 10.4% on average
[oneqb_dynasty-10] n=379  HGBR: spearman=0.838 mae=562 top50_mae=1484  |  RF baseline: spearman=0.827 mae=596 top50_mae=1729
[oneqb_dynasty-12] dropping 2 feature(s) with <2 distinct values in this group: ['breakout_age', 'dominator_rating']


[oneqb_dynasty-12] rookie quantile discount: 55 rookies in training pool, mean-model vs quantile-model prediction reduced by 7.3% on average
[oneqb_dynasty-12] n=379  HGBR: spearman=0.832 mae=566 top50_mae=1478  |  RF baseline: spearman=0.829 mae=596 top50_mae=1753
[redraft-10] dropping 2 feature(s) with <2 distinct values in this group: ['breakout_age', 'dominator_rating']


[redraft-10] rookie quantile discount: 13 rookies in training pool, mean-model vs quantile-model prediction reduced by 11.9% on average
[redraft-10] n=169  HGBR: spearman=0.749 mae=916 top50_mae=1708  |  RF baseline: spearman=0.759 mae=997 top50_mae=2003
[redraft-12] dropping 2 feature(s) with <2 distinct values in this group: ['breakout_age', 'dominator_rating']


[redraft-12] rookie quantile discount: 13 rookies in training pool, mean-model vs quantile-model prediction reduced by 9.4% on average
[redraft-12] n=169  HGBR: spearman=0.762 mae=938 top50_mae=1776  |  RF baseline: spearman=0.755 mae=1000 top50_mae=1980


,format,team_size,hgb_spearman,hgb_mae,hgb_top50_mae,rf_spearman,rf_mae
0,sf_dynasty,10,0.825811,648.982733,1574.019203,0.827224,655.211900
1,sf_dynasty,12,0.825908,671.331900,1693.519590,0.829865,658.916162
2,oneqb_dynasty,10,0.837936,562.359649,1483.791518,0.827190,596.086571
3,oneqb_dynasty,12,0.832218,565.691007,1478.093798,0.829192,596.458401
4,redraft,10,0.748991,916.329457,1708.457414,0.758863,997.177118
5,redraft,12,0.762133,937.583014,1776.299493,0.755363,999.735531


## Stage 8 -- Board assembly

Every player the model produced a final prediction for, plus (dynasty formats only) the 27 pick
assets, all on the same `model_value` scale -- the trade-calculator itself: any two assets, player
or pick, are now directly comparable.

In [11]:
boards = tcp.build_all_boards(feature_table, crosswalk, consensus_targets, pick_universe, fit_results)
for (fmt, team_size), board in boards.items():
    n_players = (board["asset_type"] == "player").sum()
    n_picks = (board["asset_type"] == "pick").sum()
    print(f"  {fmt}-{team_size}: {len(board)} rows ({n_players} players, {n_picks} picks)")

  sf_dynasty-10: 406 rows (379 players, 27 picks)
  sf_dynasty-12: 406 rows (379 players, 27 picks)
  oneqb_dynasty-10: 406 rows (379 players, 27 picks)
  oneqb_dynasty-12: 406 rows (379 players, 27 picks)
  redraft-10: 169 rows (169 players, 0 picks)
  redraft-12: 169 rows (169 players, 0 picks)


### Top 15, SF dynasty (12-team) -- eye-test sanity check

Should be recognizable star players, maybe a high pick mixed in near the very top -- **not** a
bottom-tier pick outranking Bijan Robinson.

In [12]:
boards[("sf_dynasty", 12)][["board_rank", "name", "position", "asset_type", "model_value", "consensus_value"]].head(15)

,board_rank,name,position,asset_type,model_value,consensus_value
0,1,Jahmyr Gibbs,RB,player,10362.818432,10208.844721
1,2,Bijan Robinson,RB,player,10248.266589,10405.618873
2,3,Ja'Marr Chase,WR,player,10036.008580,10095.560303
3,4,Josh Allen,QB,player,9893.008528,10215.767328
4,5,Jaxon Smith-Njigba,WR,player,8933.460219,9025.013154
5,6,Drake Maye,QB,player,8550.250195,8638.302373
6,7,Puka Nacua,WR,player,8371.281334,8389.941414
7,8,Brock Bowers,TE,player,8001.179628,7839.832817
8,9,Amon-Ra St. Brown,WR,player,7377.248456,7288.166341
9,10,Ashton Jeanty,RB,player,7223.761665,7252.992855


## Top 200 -- every board

The full trade-calculator board for each of the 6 format-teamsize combos.

### Superflex Dynasty -- 10 team -- Top 200

In [13]:
boards[("sf_dynasty", 10)][
    ["board_rank", "name", "position", "asset_type", "model_value", "consensus_value", "current_age"]
].head(200)

,board_rank,name,position,asset_type,model_value,consensus_value,current_age
0,1,Jahmyr Gibbs,RB,player,10423.362719,10347.101883,24.0
1,2,Bijan Robinson,RB,player,10279.126745,10509.841958,24.0
2,3,Ja'Marr Chase,WR,player,9921.971395,9996.099117,26.0
3,4,Josh Allen,QB,player,9635.181424,9953.403599,30.0
4,5,Jaxon Smith-Njigba,WR,player,8948.228134,8949.078601,24.0
5,6,Drake Maye,QB,player,8377.780022,8416.597602,24.0
6,7,Puka Nacua,WR,player,8293.388719,8277.130609,25.0
7,8,Brock Bowers,TE,player,7882.440078,7794.500436,23.0
8,9,Amon-Ra St. Brown,WR,player,7350.404406,7293.766710,26.0
9,10,Ashton Jeanty,RB,player,7160.420525,7297.713225,22.0


### Superflex Dynasty -- 12 team -- Top 200

In [14]:
boards[("sf_dynasty", 12)][
    ["board_rank", "name", "position", "asset_type", "model_value", "consensus_value", "current_age"]
].head(200)

,board_rank,name,position,asset_type,model_value,consensus_value,current_age
0,1,Jahmyr Gibbs,RB,player,10362.818432,10208.844721,24.0
1,2,Bijan Robinson,RB,player,10248.266589,10405.618873,24.0
2,3,Ja'Marr Chase,WR,player,10036.008580,10095.560303,26.0
3,4,Josh Allen,QB,player,9893.008528,10215.767328,30.0
4,5,Jaxon Smith-Njigba,WR,player,8933.460219,9025.013154,24.0
5,6,Drake Maye,QB,player,8550.250195,8638.302373,24.0
6,7,Puka Nacua,WR,player,8371.281334,8389.941414,25.0
7,8,Brock Bowers,TE,player,8001.179628,7839.832817,23.0
8,9,Amon-Ra St. Brown,WR,player,7377.248456,7288.166341,26.0
9,10,Ashton Jeanty,RB,player,7223.761665,7252.992855,22.0


### 1QB Dynasty -- 10 team -- Top 200

In [15]:
boards[("oneqb_dynasty", 10)][
    ["board_rank", "name", "position", "asset_type", "model_value", "consensus_value", "current_age"]
].head(200)

,board_rank,name,position,asset_type,model_value,consensus_value,current_age
0,1,Jahmyr Gibbs,RB,player,11177.318163,11168.940955,24.0
1,2,Bijan Robinson,RB,player,11133.090701,11261.907002,24.0
2,3,Ja'Marr Chase,WR,player,9975.586321,9995.442701,26.0
3,4,Jaxon Smith-Njigba,WR,player,8718.033939,8786.440989,24.0
4,5,Puka Nacua,WR,player,8249.895811,8231.556447,25.0
5,6,Ashton Jeanty,RB,player,7622.700890,7678.140830,22.0
6,7,Amon-Ra St. Brown,WR,player,7368.683118,7266.905820,26.0
7,8,Brock Bowers,TE,player,7176.475676,7188.056839,23.0
8,9,Justin Jefferson,WR,player,6685.319660,6716.844033,27.0
9,10,Malik Nabers,WR,player,6667.484779,6701.581493,23.0


### 1QB Dynasty -- 12 team -- Top 200

In [16]:
boards[("oneqb_dynasty", 12)][
    ["board_rank", "name", "position", "asset_type", "model_value", "consensus_value", "current_age"]
].head(200)

,board_rank,name,position,asset_type,model_value,consensus_value,current_age
0,1,Jahmyr Gibbs,RB,player,11059.856167,11051.886055,24.0
1,2,Bijan Robinson,RB,player,10890.779871,11143.819147,24.0
2,3,Ja'Marr Chase,WR,player,9888.263256,10036.883112,26.0
3,4,Jaxon Smith-Njigba,WR,player,8820.225227,8837.583029,24.0
4,5,Puka Nacua,WR,player,8364.843255,8279.937128,25.0
5,6,Ashton Jeanty,RB,player,7599.866936,7597.991686,22.0
6,7,Amon-Ra St. Brown,WR,player,7333.105523,7233.085765,26.0
7,8,Brock Bowers,TE,player,7309.398689,7195.210709,23.0
8,9,Justin Jefferson,WR,player,6648.424251,6756.304540,27.0
9,10,Malik Nabers,WR,player,6576.483776,6740.921188,23.0


### Redraft -- 10 team -- Top 200

In [17]:
boards[("redraft", 10)][
    ["board_rank", "name", "position", "asset_type", "model_value", "consensus_value", "current_age"]
].head(200)

,board_rank,name,position,asset_type,model_value,consensus_value,current_age
0,1,Jahmyr Gibbs,RB,player,10455.525928,10200.000000,24.0
1,2,Ja'Marr Chase,WR,player,10065.583275,9757.817630,26.0
2,3,Bijan Robinson,RB,player,9873.149942,10056.487548,24.0
3,4,Puka Nacua,WR,player,9401.861022,9436.893633,25.0
4,5,Jaxon Smith-Njigba,WR,player,9089.426194,9321.934016,24.0
5,6,Amon-Ra St. Brown,WR,player,8821.936336,8650.483990,26.0
6,7,Jonathan Taylor,RB,player,8446.587374,8348.486178,27.0
7,8,Christian McCaffrey,RB,player,8276.906555,8217.410706,30.0
8,9,CeeDee Lamb,WR,player,8090.032888,8080.936695,27.0
9,10,Ashton Jeanty,RB,player,7995.124425,8136.355457,22.0


### Redraft -- 12 team -- Top 200

In [18]:
boards[("redraft", 12)][
    ["board_rank", "name", "position", "asset_type", "model_value", "consensus_value", "current_age"]
].head(200)

,board_rank,name,position,asset_type,model_value,consensus_value,current_age
0,1,Jahmyr Gibbs,RB,player,10317.784987,10093.000000,24.0
1,2,Ja'Marr Chase,WR,player,9983.273758,9766.883164,26.0
2,3,Bijan Robinson,RB,player,9970.596929,9951.196029,24.0
3,4,Puka Nacua,WR,player,9456.952826,9492.122868,25.0
4,5,Jaxon Smith-Njigba,WR,player,9188.142385,9376.516174,24.0
5,6,Amon-Ra St. Brown,WR,player,9076.265512,8931.980820,26.0
6,7,Jonathan Taylor,RB,player,8344.475328,8260.942052,27.0
7,8,Christian McCaffrey,RB,player,8199.223681,8012.616734,30.0
8,9,CeeDee Lamb,WR,player,8073.080817,8036.159367,27.0
9,10,James Cook,RB,player,8018.147757,8074.358443,26.0


## Stage 9 -- Divergence tables

Largest model-vs-consensus gaps, player rows only (picks are priced directly from blended market
data, not model-predicted, so they always show zero residual and carry no signal here). These are
**candidates to investigate** -- fundamentals the model weighs differently than the market
currently prices them -- never a claim that the model is right and the market is wrong.

In [19]:
divergence_tables = tcp.build_all_divergence_tables(boards)
for (fmt, team_size), div_df in divergence_tables.items():
    print(f"\n=== {fmt} - {team_size} team ===")
    display(div_df)


=== sf_dynasty - 10 team ===


,name,position,model_value,consensus_value,residual,residual_pct
0,Omar Cooper Jr.,WR,2652.160150,1960.922112,691.238038,35.250663
1,C.J. Stroud,QB,3381.451811,3202.613988,178.837823,5.584120
2,Dak Prescott,QB,4083.537489,3914.681191,168.856298,4.313411
3,J.J. McCarthy,QB,1549.309291,1385.585096,163.724195,11.816250
4,Tetairoa McMillan,WR,5263.893109,5104.292371,159.600738,3.126795
5,Joe Mixon,RB,255.401529,109.736181,145.665348,132.741405
6,Harold Fannin,TE,3390.100404,3246.488983,143.611421,4.423592
7,Jack Endries,TE,207.944890,68.609399,139.335490,203.085134
8,Will Kacmarek,TE,364.689983,236.840452,127.849531,53.981290
9,Jalen Hurts,QB,5222.509149,5107.755639,114.753510,2.246652



=== sf_dynasty - 12 team ===


,name,position,model_value,consensus_value,residual,residual_pct
0,C.J. Stroud,QB,3516.672658,3258.202867,258.469791,7.932894
1,Omar Cooper Jr.,WR,2189.983172,1970.384422,219.598749,11.144970
2,J.J. McCarthy,QB,1637.469423,1423.613598,213.855825,15.022041
3,Justin Fields,QB,874.110492,685.005930,189.104562,27.606266
4,Sam Roush,TE,283.137835,100.632186,182.505649,181.359122
5,Brock Bowers,TE,8001.179628,7839.832817,161.346811,2.058039
6,Jahmyr Gibbs,RB,10362.818432,10208.844721,153.973711,1.508238
7,Breece Hall,RB,4013.540949,3863.386566,150.154383,3.886600
8,Baker Mayfield,QB,3462.280051,3313.333177,148.946874,4.495379
9,Tetairoa McMillan,WR,5143.774607,4999.390235,144.384372,2.888040



=== oneqb_dynasty - 10 team ===


...

,name,position,model_value,consensus_value,residual,residual_pct
0,Omar Cooper Jr.,WR,2380.135644,1958.291508,421.844136,21.541437
1,Germie Bernard,WR,1739.517255,1405.601977,333.915278,23.756034
2,Nate Boerkircher,TE,283.739515,127.839196,155.900319,121.950328
3,Sam Roush,TE,220.765200,85.022071,135.743129,159.656343
4,J.J. McCarthy,QB,901.990826,769.812128,132.178698,17.170254
5,Jalen Hurts,QB,3102.123428,2970.697839,131.425589,4.424065
6,Caleb Williams,QB,3928.983379,3808.495439,120.487940,3.163662
7,Dak Prescott,QB,2344.779754,2226.127204,118.652550,5.329999
8,Brock Bowers,TE,7309.398689,7195.210709,114.187980,1.587000
9,Harold Fannin,TE,3214.445082,3103.293586,111.151496,3.581727



=== redraft - 10 team ===


,name,position,model_value,consensus_value,residual,residual_pct
0,Fernando Mendoza,QB,774.368701,372.020734,402.347968,108.152028
1,Ja'Marr Chase,WR,10065.583275,9757.817630,307.765646,3.154042
2,Jahmyr Gibbs,RB,10455.525928,10200.000000,255.525928,2.505156
3,Kenneth Walker III,RB,6627.023043,6391.154245,235.868798,3.690551
4,Trey McBride,TE,6958.765651,6768.586032,190.179619,2.809739
5,Travis Etienne,RB,3043.638005,2857.231505,186.406499,6.524025
6,Bo Nix,QB,1643.069974,1456.802028,186.267946,12.786085
7,Amon-Ra St. Brown,WR,8821.936336,8650.483990,171.452347,1.981997
8,Jared Goff,QB,966.697119,820.597636,146.099483,17.804034
9,Trevor Lawrence,QB,2316.780307,2182.665154,134.115153,6.144559



=== redraft - 12 team ===


,name,position,model_value,consensus_value,residual,residual_pct
0,Fernando Mendoza,QB,787.945427,375.020734,412.924694,110.107164
1,Jahmyr Gibbs,RB,10317.784987,10093.000000,224.784987,2.227137
2,Ja'Marr Chase,WR,9983.273758,9766.883164,216.390594,2.215554
3,Christian McCaffrey,RB,8199.223681,8012.616734,186.606947,2.328914
4,Drake Maye,QB,3760.382851,3592.812416,167.570435,4.664046
5,De'Von Achane,RB,7694.688127,7540.749908,153.938219,2.041418
6,Jared Goff,QB,968.596243,820.260806,148.335436,18.083936
7,Trey McBride,TE,6845.004853,6698.266194,146.738658,2.190696
8,Amon-Ra St. Brown,WR,9076.265512,8931.980820,144.284692,1.615372
9,Kenyon Sadiq,TE,612.883860,481.232732,131.651128,27.357060


## Stage 10 -- Sanity panel

~20 fixed, real-world-unambiguous checks (absolute top-N and pairwise orderings), run every
execution, with each player's raw feature values printed alongside any failure so the "why" is
inspectable rather than just a pass/fail flag.

In [20]:
sanity_results = tcp.run_sanity_panel(boards, feature_table)
sanity_results

!!! SANITY CHECK FAILED !!! Josh Jacobs rank 69 (needed top-60) on ('oneqb_dynasty', 10)
    feature decomposition: position='RB', current_age=np.float64(28.0), career_games=np.float64(105.0), seasons_played=np.float64(7.0), ppg_ppr=np.float64(15.806666666666668), round=np.float64(1.0), pick=np.float64(24.0), dominator_rating=np.float64(nan), speed_score=np.float64(nan)

[sanity_panel] 18 passed, 1 failed, 1 skipped (player not found) of 20 checks


,check,board,passed,detail
0,Puka Nacua top-10,"(sf_dynasty, 12)",True,"rank=7 (need <= 10), model_value=8371"
1,Bijan Robinson top-5,"(sf_dynasty, 12)",True,"rank=2 (need <= 5), model_value=10248"
2,Jahmyr Gibbs top-8,"(sf_dynasty, 12)",True,"rank=1 (need <= 8), model_value=10363"
3,Ja'Marr Chase top-10,"(sf_dynasty, 12)",True,"rank=3 (need <= 10), model_value=10036"
4,Justin Jefferson top-30,"(sf_dynasty, 12)",True,"rank=15 (need <= 30), model_value=6872"
5,Amon-Ra St. Brown top-15,"(sf_dynasty, 12)",True,"rank=9 (need <= 15), model_value=7377"
6,CeeDee Lamb top-20,"(sf_dynasty, 12)",True,"rank=17 (need <= 20), model_value=6522"
7,Josh Allen top-20,"(oneqb_dynasty, 12)",True,"rank=15 (need <= 20), model_value=5726"
8,Lamar Jackson top-20,"(sf_dynasty, 12)",True,"rank=12 (need <= 20), model_value=7068"
9,Brock Bowers top-10,"(sf_dynasty, 12)",True,"rank=8 (need <= 10), model_value=8001"


## Stage 11 -- In-season update

`recompute_board(through_week=None, ...)` reproduces the preseason boards unchanged -- this is the
identity case, confirmed exactly equal below. Once the 2026 season is underway, calling
`tcp.recompute_board(through_week=<int>, feature_table, crosswalk, consensus_targets,
pick_universe, fit_results)` blends that many weeks of real partial-season production into the
feature snapshot (empirical-Bayes shrinkage against the prior-season baseline, with a `MIN_GAMES`
guard against small-sample noise) and re-predicts with the already-fitted models -- no retraining
required. Each call snapshots every board to `snapshots/{date}_{format}_{team_size}.csv` for
week-over-week movement tracking.

In [21]:
preseason_again = tcp.recompute_board(
    None, feature_table, crosswalk, consensus_targets, pick_universe, fit_results, write_snapshot=False
)
import numpy as np
same = np.allclose(
    preseason_again[("sf_dynasty", 12)]["model_value"].sort_index(),
    boards[("sf_dynasty", 12)]["model_value"].sort_index(),
)
print(f"recompute_board(through_week=None) reproduces the original boards exactly: {same}")
assert same

recompute_board(through_week=None) reproduces the original boards exactly: True


---

## Scope, exclusions, and setup

See `README.md` in this directory for full methodology, known exclusions (K/DST/IDP not modeled,
TE premium not modeled, CFBD rookie college-production features degrade gracefully when the
free-tier monthly quota is exhausted), and setup instructions.

**Setup reminder:** this notebook requires a `CFBD_API_KEY` set in a gitignored `.env` file in this
directory (`CFBD_API_KEY=your_key_here`) -- get a free key at
[collegefootballdata.com/key](https://collegefootballdata.com/key). The key is never hardcoded or
printed anywhere in this notebook or in `trade_calculator_pipeline.py`.